In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
# 1. Simple LLM call with streaming
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [ ]:
model=init_chat_model("groq:openai/gpt-oss-20b")
model

In [ ]:
# from langchain_groq import ChatGroq
# llm=ChatGroq(model="meta-llama/llama-prompt-guard-2-22m")
# llm

In [12]:
# create messages
messages=[
    SystemMessage(content="You are a helpful ai assistant"),
    HumanMessage(content="What are the top 3 benefits of using langchain?")
]

In [ ]:
# invoke the model
response=model.invoke(messages)
response

In [ ]:
print(response.content)

In [ ]:
print(model.invoke([HumanMessage("What do you mean by ML")]))

In [ ]:
# Streaming Example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

In [33]:
from langchain_core.prompts import ChatPromptTemplate

# Create translation prompt template
translation_template = ChatPromptTemplate.from_messages([
    (
        "system","You are a professional translator. Translate the following text from {source_language} to {target_language}. Maintain the tone and style."),
    ("user","{text}")
])

# Using the template
prompt = translation_template.invoke({
    "source_language": "English",
    "target_language": "Bengali",
    "text": "My name is Aditya Shaw"
})

In [ ]:
prompt

In [ ]:
translated_response=model.invoke(prompt)
print(translated_response.content)

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

def create_story_chain(model):
    # Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a creative storyteller. Write a short and engaging story based on the given theme, character, and setting."
        ),
        (
            "user",
            "Theme: {theme}\nMain character: {character}\nSetting: {setting}"
        )
    ])

    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a literary critic. Analyze the following story and provide insights."
        ),
        (
            "user",
            "{story}"
        )
    ])

    # Story generation chain
    story_chain = story_prompt | model | StrOutputParser()

    # Function to prepare the story for analysis
    def analyze_story(story_text: str) -> dict:
        return {"story": story_text}

    # Full chain: Generate story → Analyze story
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )

    return analysis_chain

In [ ]:
chain=create_story_chain(model)
chain

In [ ]:
result = chain.invoke({
    "theme":"AI",
    "character":"a curious robot",
    "setting":"a futuristic story"
})
print("Story andAnalysis:")
print(result)